마테우스 페르난데스

Defensive: Tackles 2.9, Inter 1, Fouls 1.4, Drb 1.1<br>
Offensive: KeyP 1, Drb 0.8, Disp 0.9, UnsTch 1<br>
Passing: AvgP 43.4, PS% 87.5, ThrB 0.2<br>
mins:3027

산드로 토날리

Defensive: Tackles 1, Inter 0.9, Fouls 0.7 Drb 0.7<br>
Offensive: KeyP 0.9, Drb 0.5, Disp 0.5, UnsTch 1.1 <br>
Passing: AvgP 44.7, PS% 84.4, ThrB 0.1<br>
mins: 2548

메이슨 마운트

Defensive: Tackles 1, Inter 0.4, Fouls 0.7 Drb 0.5<br>
Offensive: KeyP 0.6, Drb 0.3, Disp 0.7, UnsTch 0.8 <br>
Passing: AvgP 18.7, PS% 81.6, ThrB 0.2<br>
mins: 1020

In [ ]:
import pandas as pd

data = {
    'player': ['Mateus Fernandes', 'Sandro Tonali', 'Mason Mount'],
    'Tackles': [2.9, 1.0, 1.0],
    'Inter': [1.0, 0.9, 0.4],
    'Fouls': [1.4, 0.7, 0.7],
    'Drb_d': [1.1, 0.7, 0.5 ],
    'KeyP_o': [1.0, 0.9, 0.6],
    'Drb_o': [0.8, 0.5, 0.3],
    'Disp': [0.9, 0.5, 0.7],
    'UnsTch': [1.0, 1.1, 0.8],
    'AvgP': [43.4, 44.7, 18.7],
    'PS%': [87.5, 84.4, 81.6],
    'ThrB': [0.2, 0.1, 0.2],
    'Mins': [3027, 2548, 1020]
}

df = pd.DataFrame(data)

df['raw_수비'] = (df['Tackles'] + df['Inter']) / 2
df['raw_압박저항'] = df['Fouls']    #낮을수록 좋음 -> minmax invert
df['raw_PS%'] = df['PS%']   #높을수록 좋음 -> 따로 정규화 후 합산
df['raw_운반'] = df['Drb_o']    #높을수록 좋음
df['raw_볼손실'] = df['Disp']   #낮을수록 좋음 -> minmax invert
df['raw_전진'] = df['ThrB'] + df['KeyP_o']
df['raw_체력'] = df['Mins']

# 정규화

In [9]:
def minmax(series, invert=False):
    mn,mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    s = (series - mn) / (mx-mn)
    return (1-s) if invert else s

scores = pd.DataFrame()
scores['player'] = df['player']

#수비 안정성 - Tackles + Inter 평균, 높을수록 좋음
scores['S_수비'] = minmax(df['raw_수비'])

#압박저항 - Fouls역산 + PS% 정방향 평균
scores['S_압박저항'] = (minmax(df['raw_압박저항'], invert=True) + minmax(df['raw_PS%'])) / 2

#체력 - Mins, 높을수록 좋음
scores['S_체력'] = minmax(df['raw_체력'])

#볼 운반 - Drb_o 정방향, Disp 역산 평균
scores['S_볼 운반'] = (minmax(df['raw_볼손실'], invert=True) + minmax(df['raw_운반'])) / 2

#볼 전진 - ThrB + KeyP 합산, 높을 수록 좋음
scores['S_볼 전진'] = minmax(df['raw_전진'])

print(scores.set_index('player').round(3))

                  S_수비  S_압박저항   S_체력  S_볼 운반  S_볼 전진
player                                               
Mateus Fernandes   1.0   0.500  1.000    0.50     1.0
Sandro Tonali      0.2   0.737  0.761    0.70     0.5
Mason Mount        0.0   0.500  0.000    0.25     0.0


# 가중합으로 적합도 점수 산출

In [ ]:
WEIGHTS = {
    'S_수비': 0.30,cd
    'S_압박저항': 0.25,
    'S_체력' : 0.20,
    'S_볼 운반': 0.15,
    'S_볼 전진': 0.10
}

scores['적합도'] = (scores['S_수비']*0.30) + (scores['S_압박저항']*0.25) + (scores['S_체력']*0.20) + (scores['S_볼 운반']*0.15) + (scores['S_볼 전진']*0.10)

print(scores.set_index('player')[['적합도'] + list(WEIGHTS.keys())].sort_values('적합도', ascending=False).round(3))

                    적합도  S_수비  S_압박저항   S_체력  S_볼 운반  S_볼 전진
player                                                      
Mateus Fernandes  0.800   1.0   0.500  1.000    0.50     1.0
Sandro Tonali     0.552   0.2   0.737  0.761    0.70     0.5
Mason Mount       0.162   0.0   0.500  0.000    0.25     0.0
